In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
 
from sklearn.svm import LinearSVC
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, f1_score)
 
warnings.filterwarnings("ignore")
np.random.seed(42)
 
os.makedirs("../models", exist_ok=True)

In [ ]:
# STEP 1 — LOAD DATA
# =============================================================================
 
df = pd.read_csv("../data/raw/all_crypto_currencies.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
 
print("=" * 60)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range   : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique coins : {df['slug'].nunique()}")

In [ ]:
# STEP 2 — DATA QUALITY CHECKS
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 2 — DATA QUALITY")
 
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"\nMissing values  : {len(missing)} columns affected"
      if len(missing) else "\nMissing values  : none")
 
full_dupes = df.duplicated().sum()
key_dupes  = df.duplicated(subset=['date', 'slug']).sum()
print(f"Full duplicates : {full_dupes}")
print(f"Key duplicates  : {key_dupes}  (date + slug)")
 
empty = (df == "").sum()
empty = empty[empty > 0]
print(f"Empty strings   : {len(empty)} columns affected"
      if len(empty) else "Empty strings   : none")
 
symbol_collisions = df.groupby('symbol')['slug'].nunique()
print(f"Symbol collisions (same ticker, diff slug): "
      f"{(symbol_collisions > 1).sum()} symbols")

In [ ]:
# STEP 3 — FEATURE ENGINEERING
# All features are backward-looking per coin — no future leakage.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 3 — FEATURE ENGINEERING")
 
df = df.sort_values(['slug', 'date']).reset_index(drop=True)
grouped = df.groupby('slug', group_keys=False)
 
# --- Returns ---
df['daily_return'] = grouped['close'].pct_change()
 
# --- Moving averages ---
df['ma_30'] = (grouped['close']
               .rolling(30, min_periods=1).mean()
               .reset_index(level=0, drop=True))
 
# --- Volatility ---
df['vol_7']  = (grouped['daily_return']
                .rolling(7,  min_periods=1).std()
                .reset_index(level=0, drop=True))
df['vol_30'] = (grouped['daily_return']
                .rolling(30, min_periods=1).std()
                .reset_index(level=0, drop=True))
 
# --- Rolling range ---
df['rolling_max_7'] = (grouped['high']
                       .rolling(7, min_periods=1).max()
                       .reset_index(level=0, drop=True))
df['rolling_min_7'] = (grouped['low']
                       .rolling(7, min_periods=1).min()
                       .reset_index(level=0, drop=True))
 
# --- Lag features ---
df['lag_1'] = grouped['close'].shift(1)
df['lag_7'] = grouped['close'].shift(7)
 
# --- Momentum ---
df['momentum_7']  = df['close'] - df['lag_7']
df['momentum_14'] = df['close'] - grouped['close'].shift(14)
 
# --- Volume ---
vol_ma_7        = (grouped['volume']
                   .rolling(7, min_periods=1).mean()
                   .reset_index(level=0, drop=True))
df['vol_ratio'] = df['volume'] / (vol_ma_7 + 1e-9)
 
# --- Price spreads ---
df['high_low_spread']   = df['high'] - df['low']
df['close_open_spread'] = df['close'] - df['open']
df['high_close_ratio']  = df['high'] / (df['close'] + 1e-9)
 
# --- Market context ---
df['market_cap_ratio'] = (df['market']
                          / df.groupby('date')['market'].transform('sum'))
df['rank_normalized']  = df['ranknow'] / df['ranknow'].max()
 
# --- EMA ---
df['ema_14'] = grouped['close'].transform(
    lambda x: x.ewm(span=14, adjust=False).mean())
 
# --- RSI (14) ---
def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.rolling(period, min_periods=1).mean()
    avg_loss = loss.rolling(period, min_periods=1).mean()
    rs       = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))
 
df['rsi_14'] = grouped['close'].transform(compute_rsi)
 
# --- MACD ---
ema_12            = grouped['close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
ema_26            = grouped['close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
df['macd']        = ema_12 - ema_26
df['macd_signal'] = (grouped['macd']
                     .transform(lambda x: x.ewm(span=9, adjust=False).mean()))
 
# --- Volatility clustering ---
df['vol_cluster_14'] = (grouped['daily_return']
                        .rolling(14, min_periods=1).std()
                        .reset_index(level=0, drop=True))
 
df.fillna(0, inplace=True)
 
# --- FIX 1: Ratio-normalised features ---
# Momentum, MACD and spread in raw form are in price units — BTC momentum
# of $500 vs altcoin momentum of $0.0001 are incomparable.
# Dividing by close price makes them dimensionless ratios comparable across
# all 2,071 coins regardless of their price level.
df['macd_ratio']        = df['macd']             / (df['close'] + 1e-9)
df['macd_signal_ratio'] = df['macd_signal']       / (df['close'] + 1e-9)
df['momentum_7_ratio']  = df['momentum_7']        / (df['close'] + 1e-9)
df['momentum_14_ratio'] = df['momentum_14']       / (df['close'] + 1e-9)
df['spread_ratio']      = df['high_low_spread']   / (df['close'] + 1e-9)
df['close_open_ratio']  = df['close_open_spread'] / (df['close'] + 1e-9)
 
print(f"Feature engineering complete: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
# =============================================================================
# STEP 4 — DEFINE TARGET  (binary classification)
#
#   1 = UP   → next-day close > today's close
#   0 = DOWN → next-day close <= today's close
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 4 — TARGET DEFINITION  (binary: UP=1 / DOWN=0)")
 
df['target'] = (df.groupby('slug')['close'].shift(-1) > df['close']).astype(int)
df = df.dropna(subset=['target']).reset_index(drop=True)
 
up_pct = df['target'].mean() * 100
print(f"Rows after dropping NaNs : {df.shape[0]:,}")
print(f"Class distribution — UP (1): {up_pct:.1f}%  |  DOWN (0): {100-up_pct:.1f}%")
print("✓ Balanced" if 40 <= up_pct <= 60 else
      "⚠ Imbalanced — class_weight='balanced' applied")

In [ ]:
# STEP 5 — CHRONOLOGICAL TRAIN / VAL / TEST SPLIT  (70 / 15 / 15)
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 5 — CHRONOLOGICAL SPLIT")
 
df_sorted   = df.sort_values('date').reset_index(drop=True)
total       = len(df_sorted)
date_counts = df_sorted.groupby('date').size().reset_index(name='count')
date_counts['cum'] = date_counts['count'].cumsum()
date_counts['pct'] = date_counts['cum'] / total
 
train_end = date_counts.loc[date_counts['pct'] >= 0.70, 'date'].iloc[0]
val_end   = date_counts.loc[date_counts['pct'] >= 0.85, 'date'].iloc[0]
 
train_df = df_sorted[df_sorted['date'] <= train_end].copy()
val_df   = df_sorted[(df_sorted['date'] > train_end) & (df_sorted['date'] <= val_end)].copy()
test_df  = df_sorted[df_sorted['date'] > val_end].copy()
 
print(f"Train : {len(train_df):>8,} rows  (up to {train_end.date()})")
print(f"Val   : {len(val_df):>8,} rows  (up to {val_end.date()})")
print(f"Test  : {len(test_df):>8,} rows  (after {val_end.date()})")

In [ ]:
# STEP 6 — FEATURE SELECTION  (scale-invariant features only)
#
# FIX 1: Only features that are comparable across all coin price levels.
# Excluded: lag_1, lag_7, ma_30, ema_14, rolling_max_7, rolling_min_7,
#           momentum_7, momentum_14, macd, macd_signal, high_low_spread,
#           close_open_spread — all are in price units, incomparable across coins.
# Kept:    ratio/percentage features + their normalised equivalents.
# =============================================================================
 
SCALE_FREE_FEATURES = [
    'daily_return',       # % return — same meaning for any coin
    'close_ratio',        # position within daily high-low range (0 to 1)
    'vol_7',              # 7-day return volatility — scale free
    'vol_30',             # 30-day return volatility — scale free
    'vol_ratio',          # today's volume vs 7-day avg — dimensionless
    'vol_cluster_14',     # 14-day volatility clustering — scale free
    'rsi_14',             # RSI always 0–100 regardless of price
    'high_close_ratio',   # high / close — dimensionless ratio
    'macd_ratio',         # MACD normalised by price
    'macd_signal_ratio',  # MACD signal normalised by price
    'momentum_7_ratio',   # 7-day momentum normalised by price
    'momentum_14_ratio',  # 14-day momentum normalised by price
    'spread_ratio',       # high-low range normalised by price
    'close_open_ratio',   # intraday move normalised by price
    'market_cap_ratio',   # share of total market cap — already normalised
    'rank_normalized',    # market rank 0–1 — already normalised
]
 
print("\n" + "=" * 60)
print(f"STEP 6 — FEATURE SELECTION  ({len(SCALE_FREE_FEATURES)} scale-invariant features)")
print(SCALE_FREE_FEATURES)
 
X_train_raw = train_df[SCALE_FREE_FEATURES].values
X_val_raw   = val_df[SCALE_FREE_FEATURES].values
X_test_raw  = test_df[SCALE_FREE_FEATURES].values
y_train     = train_df['target'].values
y_val       = val_df['target'].values
y_test      = test_df['target'].values
 
# Clean any inf/nan introduced by division
X_train_raw = np.nan_to_num(X_train_raw, nan=0.0, posinf=0.0, neginf=0.0)
X_val_raw   = np.nan_to_num(X_val_raw,   nan=0.0, posinf=0.0, neginf=0.0)
X_test_raw  = np.nan_to_num(X_test_raw,  nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
# STEP 7 — SCALING + PCA
#
# FIX 2: RobustScaler handles crypto outliers better than StandardScaler.
# FIX 3: PCA concentrates diffuse signal into fewer strong components.
#         SVM draws a cleaner boundary through concentrated dimensions
#         than through 16 weakly-correlated features simultaneously.
#
# Both scaler and PCA are fit on train only — no leakage into val/test.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 7 — SCALING + PCA")
 
scaler = RobustScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_val_sc   = scaler.transform(X_val_raw)
X_test_sc  = scaler.transform(X_test_raw)
 
N_COMPONENTS = 10
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train_sc)
X_val_pca   = pca.transform(X_val_sc)
X_test_pca  = pca.transform(X_test_sc)
 
explained = pca.explained_variance_ratio_.cumsum()
print(f"PCA: {N_COMPONENTS} components explain {explained[-1]*100:.1f}% of variance")
print(f"Variance per component: "
      f"{[f'{v*100:.1f}%' for v in pca.explained_variance_ratio_]}")

In [ ]:
# STEP 8 — TRAIN LinearSVC WITH C GRID SEARCH
#
# FIX 4: LinearSVC trains on the FULL 660k training set.
#         No subsampling needed — liblinear solver is O(n), not O(n²).
# FIX 5: C grid search on val ROC-AUC — proper hyperparameter selection.
#
# CalibratedClassifierCV wraps LinearSVC to produce probability estimates
# needed for ROC-AUC. Uses 3-fold cross-validation for calibration.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 8 — C GRID SEARCH  (LinearSVC, full training set)")
 
baseline_acc = max(y_val.mean(), 1 - y_val.mean()) * 100
print(f"\n  Baseline (majority class): {baseline_acc:.2f}%")
print(f"\n  {'C':>8}  {'Val Acc':>10}  {'ROC-AUC':>10}  "
      f"{'DOWN recall':>12}  {'UP recall':>10}  {'Macro F1':>10}")
print(f"  {'─'*66}")
 
grid_results = []
 
for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    model = CalibratedClassifierCV(
                LinearSVC(C=C, max_iter=3000,
                          class_weight='balanced',
                          random_state=42),
                cv=3)
    model.fit(X_train_pca, y_train)
 
    preds  = model.predict(X_val_pca)
    proba  = model.predict_proba(X_val_pca)[:, 1]
    acc    = accuracy_score(y_val, preds) * 100
    auc    = roc_auc_score(y_val, proba)
    f1     = f1_score(y_val, preds, average='macro', zero_division=0)
    report = classification_report(y_val, preds,
                                   output_dict=True, zero_division=0)
    dr     = report['0']['recall']
    ur     = report['1']['recall']
 
    print(f"  {C:>8}  {acc:>9.2f}%  {auc:>10.4f}  "
          f"{dr:>12.3f}  {ur:>10.3f}  {f1:>10.4f}")
 
    grid_results.append({
        'C': C, 'val_acc': acc, 'val_auc': auc,
        'val_f1': f1, 'down_recall': dr, 'up_recall': ur,
        'model': model
    })
 
# Select best by ROC-AUC
best        = max(grid_results, key=lambda x: x['val_auc'])
best_model  = best['model']
best_C      = best['C']
 
print(f"\n  Best C       : {best_C}")
print(f"  Best ROC-AUC : {best['val_auc']:.4f}")
print(f"  Best Macro F1: {best['val_f1']:.4f}")

In [ ]:
# STEP 9 — THRESHOLD CALIBRATION
#
# FIX 6: Default threshold of 0.5 causes the model to predict only one class.
#         We search thresholds 0.1–0.9 and pick the one that maximises
#         macro F1 on the validation set — ensuring both classes are predicted.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 9 — THRESHOLD CALIBRATION")
 
val_proba_up = best_model.predict_proba(X_val_pca)[:, 1]
thresholds   = np.arange(0.10, 0.90, 0.01)
 
f1_scores    = []
down_recalls = []
up_recalls   = []
accs         = []
 
for t in thresholds:
    preds_t = (val_proba_up >= t).astype(int)
    f1_scores.append(f1_score(y_val, preds_t, average='macro', zero_division=0))
    accs.append(accuracy_score(y_val, preds_t))
    rpt = classification_report(y_val, preds_t, output_dict=True, zero_division=0)
    down_recalls.append(rpt['0']['recall'])
    up_recalls.append(rpt['1']['recall'])
 
f1_scores      = np.array(f1_scores)
best_idx       = f1_scores.argmax()
best_threshold = thresholds[best_idx]
 
print(f"\n  Best threshold (max macro F1) : {best_threshold:.2f}")
print(f"  Macro F1 at best threshold    : {f1_scores[best_idx]:.4f}")
print(f"  DOWN recall at threshold      : {down_recalls[best_idx]:.3f}")
print(f"  UP recall at threshold        : {up_recalls[best_idx]:.3f}")
 
# Plot
plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1_scores,    label='Macro F1',    linewidth=2)
plt.plot(thresholds, accs,         label='Accuracy',    linewidth=2)
plt.plot(thresholds, down_recalls, label='DOWN recall', linewidth=1.5, linestyle='--')
plt.plot(thresholds, up_recalls,   label='UP recall',   linewidth=1.5, linestyle='--')
plt.axvline(best_threshold, color='red', linestyle=':',
            label=f'Best threshold ({best_threshold:.2f})')
plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Threshold vs Metrics — Validation Set")
plt.legend()
plt.tight_layout()
plt.savefig("../models/svm_threshold_calibration.png", dpi=150)
plt.show()

In [ ]:
# STEP 10 — EVALUATE AT OPTIMAL THRESHOLD  (train + val)
# Test set still held out.
# =============================================================================
 
print("\n" + "=" * 60)
print(f"STEP 10 — EVALUATION AT THRESHOLD = {best_threshold:.2f}")
 
def evaluate_at_threshold(model, X, y, threshold, label):
    proba = model.predict_proba(X)[:, 1]
    preds = (proba >= threshold).astype(int)
    acc   = accuracy_score(y, preds) * 100
    auc   = roc_auc_score(y, proba)
    f1    = f1_score(y, preds, average='macro', zero_division=0)
    print(f"\n  {label}")
    print(f"    Accuracy : {acc:.2f}%")
    print(f"    ROC-AUC  : {auc:.4f}")
    print(f"    Macro F1 : {f1:.4f}")
    print(f"\n{classification_report(y, preds, target_names=['DOWN (0)', 'UP (1)'], digits=3, zero_division=0)}")
    return preds, proba
 
train_preds, train_proba = evaluate_at_threshold(
    best_model, X_train_pca, y_train, best_threshold, "Train (full)")
val_preds, val_proba = evaluate_at_threshold(
    best_model, X_val_pca,   y_val,   best_threshold, "Validation")
 
gap = accuracy_score(y_train, train_preds) - accuracy_score(y_val, val_preds)
print(f"  Overfit gap (Train Acc - Val Acc): {gap*100:.2f}%")
print("  ✓ Gap acceptable" if abs(gap*100) <= 5 else "  ⚠ Overfit — reduce C")

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

pipeline = {
    'scaler': scaler,
    'pca': pca,
    'model': best_model,
    'threshold': best_threshold,
    'features': SCALE_FREE_FEATURES
}

joblib.dump(pipeline, "../models/svm_model.pkl")
print("SVM pipeline saved successfully.")